In [1]:
puts `ls -l rawdata`

total 7716
-rw-rw-r-- 1 osboxes osboxes 3678593 Aug 12 17:45 disease-gene.csv
-rw-rw-r-- 1 osboxes osboxes 1419434 Aug 12 17:45 disease-therapeuticarea.csv
-rw-rw-r-- 1 osboxes osboxes   50286 Aug 12 17:45 drug-disease.csv
-rw-rw-r-- 1 osboxes osboxes   97134 Aug 12 17:45 drug-drugtype.csv
-rw-rw-r-- 1 osboxes osboxes  148014 Aug 12 17:45 drug-gene.csv
-rw-rw-r-- 1 osboxes osboxes     576 Aug 12 17:45 edges.csv
-rw-rw-r-- 1 osboxes osboxes  201313 Aug 12 17:45 gene-reactome.csv
-rw-rw-r-- 1 osboxes osboxes 2282825 Aug 12 17:45 Graph_edge_info.xlsx


In [8]:
puts `head -1 ./rawdata/drug-disease.csv`
#puts
puts `grep 'HP_' ./rawdata/drug-disease.csv | head -20`

"source","source_type","target","target_type"
"CHEMBL1231","drug","HP_0000103","disease"
"CHEMBL364713","drug","HP_0012735","disease"
"CHEMBL762","drug","HP_0001742","disease"
"CHEMBL112","drug","HP_0100607","disease"
"CHEMBL2110805","drug","HP_0000103","disease"
"CHEMBL1201460","drug","HP_0004419","disease"
"CHEMBL1201666","drug","HP_0004419","disease"
"CHEMBL2108222","drug","HP_0004419","disease"
"CHEMBL1201476","drug","HP_0004419","disease"
"CHEMBL1286","drug","HP_0002069","disease"
"CHEMBL2108147","drug","HP_0004419","disease"
"CHEMBL1200878","drug","HP_0001742","disease"
"CHEMBL1200878","drug","HP_0006536","disease"
"CHEMBL1518","drug","HP_0000820","disease"
"CHEMBL1536","drug","HP_0002148","disease"
"CHEMBL1615438","drug","HP_0001742","disease"
"CHEMBL1200640","drug","HP_0001891","disease"
"CHEMBL2010507","drug","HP_0006536","disease"
"CHEMBL636","drug","HP_0000726","disease"
"CHEMBL88055","drug","HP_0006536","disease"


In [2]:
puts `ls -lt maps`

total 960
-rw-rw-r-- 1 osboxes osboxes  16053 Feb 17 15:05 no_mondo_mapping.txt
-rw-rw-r-- 1 osboxes osboxes  92125 Jan 22 10:05 drugs.map
-rw-rw-r-- 1 osboxes osboxes   2029 Jan 22 10:05 drugs-biologics-errors.txt
-rw-rw-r-- 1 osboxes osboxes  55707 Jan 21 18:53 diseases-errors.txt
-rw-rw-r-- 1 osboxes osboxes 646476 Jan 21 18:53 diseases.map
-rw-rw-r-- 1 osboxes osboxes  11009 Jan  7 13:20 drugs-errors.txt
-rw-rw-r-- 1 osboxes osboxes 135160 Jan  7 13:20 genes.map
drwxrwxr-x 2 osboxes osboxes   4096 Jan  7 13:20 backup_pre_NYE
drwxrwxr-x 2 osboxes osboxes   4096 Jan  7 13:20 DEPRECATED


In [3]:
puts `head -2 ./maps/drugs.map`

chembl,label,CID,IUPACname
CHEMBL1200656,CHEMBL1200656,5284447,Natamycin


### note that the CID column will be prefixed with "SUBSTANCE_" for biologics, and the URL should be /substances/
```
CHEMBL1201632,CHEMBL1201632,SUBSTANCE_50069655,Cerezyme
CHEMBL2108078,CHEMBL2108078,SUBSTANCE_135651867,Natpar
```

In [4]:
puts `head -2 ./maps/diseases.map`

source,mondo,prefname
EFO_0000174,http://purl.obolibrary.org/obo/MONDO_0012817,Ewing sarcoma


In [2]:
puts `grep Orphanet ./maps/diseases.map | head -20`

Orphanet_10,http://purl.obolibrary.org/obo/MONDO_0015028,"48,XXYY syndrome"
Orphanet_100,http://purl.obolibrary.org/obo/MONDO_0008840,Ataxia-telangiectasia
Orphanet_1000,http://purl.obolibrary.org/obo/MONDO_0010390,Ocular albinism with late-onset sensorineural deafness
Orphanet_100006,http://purl.obolibrary.org/obo/MONDO_0015033,"ABeta amyloidosis, Dutch type"
Orphanet_100008,http://purl.obolibrary.org/obo/MONDO_0007098,ACys amyloidosis
Orphanet_100011,http://purl.obolibrary.org/obo/MONDO_0015034,Lissencephaly with cerebellar hypoplasia type A
Orphanet_100012,http://purl.obolibrary.org/obo/MONDO_0015035,Lissencephaly with cerebellar hypoplasia type B
Orphanet_100013,http://purl.obolibrary.org/obo/MONDO_0015036,Lissencephaly with cerebellar hypoplasia type C
Orphanet_100014,http://purl.obolibrary.org/obo/MONDO_0015037,Lissencephaly with cerebellar hypoplasia type D
Orphanet_100015,http://purl.obolibrary.org/obo/MONDO_0015038,Lissencephaly with cerebellar hypoplasia type E
Orphanet_10001

In [27]:
require 'net/http'
require 'json'
require 'uri'

# Fetch the rdfs:label for an HPO term via EBI OLS4 (authoritative mirror, always current).
# hp_code: "HP:0001250" or "HP_0001250"
require 'net/http'
require 'json'
require 'uri'

def get_hpo_label(hp_code)
  local   = hp_code.tr(':', '_').then { |s| s.start_with?('HP_') ? s : "HP_#{s}" }
  iri     = "http://purl.obolibrary.org/obo/#{local}"
  encoded = URI.encode_uri_component(URI.encode_uri_component(iri))
  uri     = URI("https://www.ebi.ac.uk/ols4/api/ontologies/hp/terms/#{encoded}")

  response = Net::HTTP.get_response(uri)
  return "no HPO match found for #{local}" unless response.is_a?(Net::HTTPSuccess)

  data = JSON.parse(response.body)
  data['label'] || "no HPO match found for #{local}"
rescue => e
  "no HPO match found for #{local}"
end

puts get_hpo_label('HP:0001250')   # => "Seizure"
puts get_hpo_label('HP_0001250')   # also works

Seizure
Seizure


# Some diseases are modelled as EFO.  These have been deprecated bu EBI-UK and replaced by MONDO.

This code DOES NOT handle these deprecated classes

In [28]:
require 'linkeddata'
require 'rdf/nquads'
require 'csv'

e = File.open('./graph/drug-phenotype-errors.txt', 'w') 

# Define namespaces
SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')

# Read input files
drug_mappings = CSV.read('./maps/drugs.map', headers: true)

failures = {}

# refresh
f = File.open('./graph/radboud_drug-phenotype.nq.large', 'w')
f.close


CSV.foreach('./rawdata/drug-disease.csv', col_sep: ",", quote_char: '"', 
  liberal_parsing: true, headers: true) do |row|
# "source","source_type","target","target_type"
# "CHEMBL1231","drug","HP_0000103","disease"
# "CHEMBL364713","drug","HP_0012735","disease"
  hpo_num = row['target']
  next unless hpo_num.match(/HP_/)  # phenos only

  drug_id = row['source']
#   warn "searching for #{drug_id}"
  # chembl,label,CID,IUPACname
  # CHEMBL1200656,CHEMBL1200656,5284447,Natamycin
  # CHEMBL1201632,CHEMBL1201632,SUBSTANCE_50069655,Cerezyme
  drug = drug_mappings.find { |d| d['chembl'] == drug_id }
  
#     warn "found #{drug}"
  unless drug
    next if failures[drug_id]
    failures[drug_id] = 1
    warn "drug lookup failed #{drug_id}"
    e.write "drug lookup failed #{drug_id}\n"
    next
  end
  
  # Extract relevant IDs and labels
  # chembl,label,CID,IUPACname
  # CHEMBL1200656,CHEMBL1200656,5284447,Natamycin
  # CHEMBL1201632,CHEMBL1201632,SUBSTANCE_50069655,Cerezyme
  drug_id = drug['CID']
  if drug_id =~ /SUBSTANCE_(\d+)/
      pubchem_uri = RDF::URI.new("https://pubchem.ncbi.nlm.nih.gov/substance/#{$1}")
  else
      pubchem_uri = RDF::URI.new("https://pubchem.ncbi.nlm.nih.gov/compound/#{drug['CID']}")
  end
  pubchem_type = RDF::URI.new("http://semanticscience.org/resource/CHEMINF_000302")
  pubchem_label =  RDF::Literal.new("PubChem Identifier")
  pubchem_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Drug")
  drug_label = RDF::Literal.new(drug['label'])
  iupac_drug_label = RDF::Literal.new(drug['IUPACname'])
  original_drug = RDF::Literal.new(drug['chembl'])



  hpo      = RDF::URI.new("http://purl.obolibrary.org/obo/#{hpo_num}")
  hpo_type      = RDF::URI.new("http://edamontology.org/data_3275")
  hpo_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Phenotype") 
  hpo_label = RDF::Literal.new(get_hpo_label(hpo_num))
  original_disease = RDF::Literal.new(hpo_num)
  
  # Create context URI
  context_uri = RDF::URI.new("urn:simpathic:context:#{drug_id}_#{hpo_num}")
  general_context = RDF::URI.new("urn:simpathic:context:all_metadata")
  
  # Create RDF repository (need to do this each time, since there are hundreds of thousands of lines, and the graph gets too big for memory)
  graph = RDF::Repository.new

  # Add quads to graph using RDF::Statement
  graph << RDF::Statement.new(pubchem_uri,     SIMPATHIC['associated-with'], hpo,          graph_name: context_uri)
  graph << RDF::Statement.new(hpo,             SIMPATHIC['associated-with'], pubchem_uri,  graph_name: context_uri)
  
  # graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     drug_label, graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,    RDFS.label,     iupac_drug_label,            graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,    RDF.type,       pubchem_type,                graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,    RDF.type,       pubchem_core_type,           graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_type,   RDFS.label,     RDF::Literal.new("PubChem"), graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_core_type, RDFS.label,  RDF::Literal.new("Drug"),    graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,    SIMPATHIC['original-id'], original_drug,     graph_name: context_uri)

      
    #  THIS NEEDS TO BE DELETED FROM THE DB
    # graph << RDF::Statement.new(hpo,          RDFS.label,               RDF::Literal.new("HPO Phenotype Identifier"), graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          RDFS.label,               hpo_label,                                    graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          RDF.type,                 hpo_type,                                     graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          RDF.type,                 hpo_core_type,                                graph_name: context_uri)
    graph << RDF::Statement.new(hpo_type,     RDFS.label,               RDF::Literal.new("HPO Ontology Term"),        graph_name: context_uri)
    graph << RDF::Statement.new(hpo_core_type,RDFS.label,               RDF::Literal.new("Phenotype"),                graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          SIMPATHIC['original-id'], original_disease,                             graph_name: context_uri)

    
    graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'],       RDF::Literal.new("Radboud"),     graph_name: general_context)
    # graph << RDF::Statement.new(context_uri, SIMPATHIC['evidence'],         RDF::URI.new(evidence),                   graph_name: general_context)
    # graph << RDF::Statement.new(context_uri, SIMPATHIC['score'],            RDF::Literal.new(score),                  graph_name: general_context)
    graph << RDF::Statement.new(context_uri, SIMPATHIC['source-relation'],  RDF::Literal.new("TREAT"),      graph_name: general_context)
    graph << RDF::Statement.new(context_uri, SIMPATHIC['source-relation'],  RDF::Literal.new("TREAT_EXPANSIVE"),      graph_name: general_context)
  
  # Write RDF to file in N-Quads format
  File.open('./graph/radboud_drug-phenotype.nq.large', 'a') do |f|
    RDF::Writer.for(:nquads).new(f) do |writer|
      writer << graph
    end
  end
end

e.close

puts "RDF quads written"

(irb):7: warning: already initialized constant Object::SIMPATHIC
(irb):7: warning: previous definition of SIMPATHIC was here
(irb):8: warning: already initialized constant Object::RDFS
(irb):8: warning: previous definition of RDFS was here
drug lookup failed CHEMBL2108222
drug lookup failed CHEMBL1200640
drug lookup failed CHEMBL1201657
drug lookup failed CHEMBL2103749
drug lookup failed CHEMBL604
drug lookup failed CHEMBL1201589


RDF quads written


In [29]:
puts `cat ./graph/radboud_drug-phenotype.nq.large | wc -l`
puts `head -40 ./graph/radboud_drug-phenotype.nq.large`

5729
<https://pubchem.ncbi.nlm.nih.gov/compound/4634> <urn:simpathic:associated-with> <http://purl.obolibrary.org/obo/HP_0000103> <urn:simpathic:context:4634_HP_0000103> .
<https://pubchem.ncbi.nlm.nih.gov/compound/4634> <http://www.w3.org/2000/01/rdf-schema#label> "Oxybutynin" <urn:simpathic:context:4634_HP_0000103> .
<https://pubchem.ncbi.nlm.nih.gov/compound/4634> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://semanticscience.org/resource/CHEMINF_000302> <urn:simpathic:context:4634_HP_0000103> .
<https://pubchem.ncbi.nlm.nih.gov/compound/4634> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <https://w3id.org/biolink/vocab/Drug> <urn:simpathic:context:4634_HP_0000103> .
<https://pubchem.ncbi.nlm.nih.gov/compound/4634> <urn:simpathic:original-id> "CHEMBL1231" <urn:simpathic:context:4634_HP_0000103> .
<http://purl.obolibrary.org/obo/HP_0000103> <urn:simpathic:associated-with> <https://pubchem.ncbi.nlm.nih.gov/compound/4634> <urn:simpathic:context:4634_HP_0000103> .
<http:/